# Roundoff error and finite precision representations

&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/ExecutableEngineering/blob/main/chapters/numerical_error/roundoff_error_and_finite_precision_representations.ipynb)



In [1]:
import sys
import os

# 1. Define the path to the local source code
local_package_path = os.path.abspath('../../executable_engineering')
if os.path.exists(local_package_path):
    sys.path.insert(0, local_package_path)


In [2]:
import numpy as np
import executable_engineering as exe

**Round-off errors** are an unavoidable consequence of representing numbers with a finite number of digits. This affects both irrational numbers and rational numbers that cannot be perfectly represented in a given *base*.

In [46]:
print(f"Pi: {np.pi}")
print(f"e: {np.e}")
print(f"1/3: {1/3}")
print(f"sqrt(2): {np.sqrt(2)}")

Pi: 3.141592653589793
e: 2.718281828459045
1/3: 0.3333333333333333
sqrt(2): 1.4142135623730951


Because memory is finite, computers must round or truncate numbers. While a single rounding error is tiny, these small errors can accumulate over millions of computations and lead to massive discrepancies.

> The average human is capable of around one mistake per second, but computers can make millions of mistakes a second!

The magnitude of round-off error is influenced by the **precision** (the number of digits used) and the **base** of the number system.

## Binary Representation


Humans typically use a base-10 (**decimal**) numbering system, likely because we have ten fingers. 

> An exception to this is the Mayans, who used a base-20 (vigesimal) system.

In the decimal system, each digit's position corresponds to a power of 10. For example, 1305 can be expressed as:

$1305_{10} = 5 \times 10^0 + 0 \times 10^1 + 3 \times 10^2 + 1 \times 10^3$


Computers use a base-2 (**binary**) system, where each bit is either 0 or 1. The same number, 1305, is written in binary as:


In [4]:
np.binary_repr(1305)

'10100011001'

We can verify this by converting the binary representation back to decimal:

$$
\begin{aligned}
10100011001_2 &= 1 \times 2^0 + 0 \times 2^1 + 0 \times 2^2 + 1 \times 2^3 + 1 \times 2^4 \\
&\quad + 0 \times 2^5 + 0 \times 2^6 + 0 \times 2^7 + 1 \times 2^8 + 0 \times 2^9 + 1 \times 2^{10} \\
&= 1305_{10}
\end{aligned}
$$ (binary_expansion)


Binary representation can also include fractional parts using powers of 2 (e.g. $2^{-1}$, $2^{-2}$). For example, $54.75_{10}$ can be written in binary:


In [5]:
def decimal_to_binary(num, precision=20):
    # 1. Handle negative numbers correctly
    sign = "-" if num < 0 else ""
    num = abs(num)
    
    # 2. Split into integer and fractional parts cleanly
    int_part = int(num)
    frac = num - int_part
    
    # 3. Convert integer part using Python's built-in bin() 
    int_str = bin(int_part)[2:]
    
    # 4. Convert fractional part
    frac_part = ""
    for _ in range(precision):
        if frac == 0: 
            break
        frac *= 2
        bit = int(frac)
        frac_part += str(bit)
        frac -= bit
        
    # 5. Handle cases with no fractional part
    if not frac_part:
        frac_part = "0"
        
    # --- PRINT THE DETAILED BREAKDOWN ---
    print(f"sign = '{sign}'")
    print(f"int_str = '{int_str}'")
    print(f"frac_part = '{frac_part}'")
    # ------------------------------------
        
    return f"{sign}{int_str}.{frac_part}"

# Test it out:
print(f"Final result for 54.75: {decimal_to_binary(54.75)}\n")
print(f"Final result for -0.1: {decimal_to_binary(-0.1)}\n")
print(f"-54.75 in binary is: {decimal_to_binary(-54.75)}\n")

sign = ''
int_str = '110110'
frac_part = '11'
Final result for 54.75: 110110.11

sign = '-'
int_str = '0'
frac_part = '00011001100110011001'
Final result for -0.1: -0.00011001100110011001

sign = '-'
int_str = '110110'
frac_part = '11'
-54.75 in binary is: -110110.11



Check:

$$
\begin{aligned}
110110.11_2 &= (1 \times 2^5 + 1 \times 2^4 + 0 \times 2^3 + 1 \times 2^2 + 1 \times 2^1 + 0 \times 2^0) \\
&\quad + (1 \times 2^{-1} + 1 \times 2^{-2}) \\
&= 54.75_{10}
\end{aligned}
$$ (binary_fraction)


In [6]:
# Examples:
print(f"54.75 in memory: {exe.python_internal_binary(54.75)}")
print(f"0.1 in memory:   {exe.python_internal_binary(0.1)}")
print(f"-54.75 in memory: {exe.python_internal_binary(-54.75)}")

54.75 in memory: 0 10000000100 1011011000000000000000000000000000000000000000000000
0.1 in memory:   0 01111111011 1001100110011001100110011001100110011001100110011010
-54.75 in memory: 1 10000000100 1011011000000000000000000000000000000000000000000000


### Example: Convert 0.1 to Binary


In [7]:
print(f"0.1 in binary is: {decimal_to_binary(0.1)}")

sign = ''
int_str = '0'
frac_part = '00011001100110011001'
0.1 in binary is: 0.00011001100110011001


The binary representation of $0.1$ is actually a repeating fraction, which means it cannot be perfectly represented with a finite number of bits.


## Precision


Computers store data in units called *words*. The number of bits in a word determines its **precision**. The IEEE standard defines the following precisions:

| Precision | Number of Bits |
|-----------|----------------|
| Single    | 32             |
| Double    | 64             |
| Quad      | 128            |

For comparison, the binary representation of 1305 (`10100011001`) requires 11 bits. Modern computers typically use double precision (64-bit) for floating-point calculations, which is the default in Python 3.


## Integers


Integers are used for signed numbers and **do not suffer from round-off error**. However, they have a limited absolute range bounded by the number of bits used to store them.

The range is determined by $2^{\text{bits}}$ and split between positive and negative values. Due to the representation of zero, the negative range is slightly larger.

The minimum and maximum values for a signed integer are:

$min = -2^{\text{bits}-1}$

$max = 2^{\text{bits}-1} - 1$

> Modern computers use a method called *Two's Complement* to represent signed integers, rather than a simple sign bit.


### Example: What is the largest integer a double-precision variable can store?


In [8]:
print(f"min: {-2**63}")
print(f"max: {2**63-1}")

min: -9223372036854775808
max: 9223372036854775807


We can check with the built-in numpy examiner:


In [9]:
print(np.iinfo(np.int64))

Machine parameters for int64
---------------------------------------------------------------
min = -9223372036854775808
max = 9223372036854775807
---------------------------------------------------------------



### Example 2: Overflow error


What happens when you store a number too large? 


In [10]:
 # works because 2**62 is within the range of a 64-bit integer
print(np.int64(2**62))

# This will cause an overflow error because 2**63 is too large
print(np.int64(2**63))

4611686018427387904


OverflowError: int too big to convert

Let's try some more.


In [11]:
# This also works
print(np.int64(1000000000000000000))

# This will also cause an overflow error
print(np.int64(10000000000000000000))

1000000000000000000


OverflowError: int too big to convert

### Example 3 Operations
What do you think the answers to these operations are?


In [16]:
a = np.int64(4)
b = np.int64(3)

print(a+b)
print(a-b)
print(a*b)
print(a/b)

7
1
12
1.3333333333333333


Were you expecting $a/b$ to result in 1? Python is sophisticated enough to handle this intelligently but other languages optimized for speed might not. Beware!

## Floating-Point Numbers


Writing out very large or small numbers is impractical. It is much more efficient to use scientific notation to represent the magnitude as an exponent:

$10,000,000,000,000,000,000 = 10^{19}$


### Floating-Point Decimal Numbers (Scientific Notation)


We can remove placeholder zeros by using a *floating point* to separate the fractional part (mantissa) from the order of magnitude (exponent).

**Scientific Notation:** $mantissa \times 10^{exponent}$

| Decimal      | Scientific Notation     | Mantissa | Exponent |
|--------------|--------------------------|----------|----------|
| $265.73$     | $2.6573 \times 10^2$    | 2.6573   | 2        |
| $0.0001$     | $1 \times 10^{-4}$        | 1        | -4       |
| $-0.0034123$ | $-3.4123 \times 10^{-3}$ | -3.4123  | -3       |
| $1500^*$     | $1.5 \times 10^3$       | 1.5      | 3        |

*Assuming the trailing zeros are not significant.

**Note:**
1. The mantissa is a fraction. If we demand that the decimal point be after the first digit, we can drop the decimal point and represent the manittisa as an integer.
2. The exponent is the power of the number system's base (in this case, 10).


In [17]:
def to_scientific(num):
    if num == 0: return "0"
    exp = 0
    while abs(num) < 1:
        num *= 10; exp -= 1
    while abs(num) >= 10:
        num /= 10; exp += 1
    return f"{num}E{exp}"

for val in [265.73, 0.0001, -0.0034123, 1500, 0]:
    print(to_scientific(val))

2.6573E2
1.0E-4
-3.4123E-3
1.5E3
0


### Floating-Point Binary Numbers


The same floating-point concept can be applied to binary numbers using base 2:

$mantissa \times 2^{exponent}$


#### Example: Convert 54.75 into Floating-Point Binary


$54.75_{10} = 110110.11_2$

To normalize this, we move the binary point so that it is after the first non-zero digit:

$1.1011011_2 \times 2^5$

The exponent is 5, which in binary is $101_2$. So the full floating-point representation is:

$1.1011011_2 \times 2^{101_2}$


### Precision in Floating-Point Numbers


Since we are limited by a finite number of bits, floating-point numbers are divided into three precision parts according to the IEEE 754 standard:

| Precision | Total Bits | Sign | Exponent | Mantissa |
|:----------|:-----------|:-----|:---------|:---------|
| Single    | 32         | 1    | 8        | 23       |
| Double    | 64         | 1    | 11       | 52       |
| Quad      | 128        | 1    | 15       | 112      |

The **sign** bit determines if the number is positive or negative. The **exponent** bits store the magnitude, and the **mantissa** stores the fractional part.


### Example: How is 0.1 actually stored?


In [22]:
print(format(0.1, '.55f'))
# Try 0.10000000000000000000000000000000000000000000000

0.1000000000000000055511151231257827021181583404541015625


As you can see, the stored value is not exactly 0.1. This is because 0.1 is (surprinsingly) a repeating fraction in binary and *cannot* be perfectly represented in finite precision.


## Round-off error


**Roundoff error** is the difference between the true number and the finite-precision representation. This error can be mitigated by using higher precision at the cost of computational speed and memory.

>Roundoff errors are important to keep in mind when working with numbers of ?similar or vastly different scales.


### Different magnitudes of numbers


Finite precision means small numbers can be completely lost when added to very large numbers. Because of this, we cannot always rely on the *associative* property of addition:


In [ ]:
print(f"-1 + (1 + 1e-20) = {-1+(1+1e-20)}")
print(f"(-1 + 1) + 1e-20 = {(-1+1)+1e-20}")
# What about -1 + 1 + 1e-20? 
# What about -1 + 1e-20 + 1? 

-1 + (1 + 1e-20) = 0.0
(-1 + 1) + 1e-20 = 1e-20
0.0


Although individual errors are small, algorithms involve many steps and errors can easily accumulate. Compilers and simplification steps may mitigate these errors, but it is often best to rely on packaged subroutines that are *numerical stabilized* - specifically to avoid such problems. 

> NB: This is a case of user discretion when using GenAI methods which may be overly eager to reinvent the wheel!

### Subtractive cancellation


**Subtractive cancellation** happens when two nearly equal numbers are subtracted, causing a dramatic loss of significant digits. 


In [27]:
a = np.float32(1.23456789)
b = np.float32(1.23456780)
res = a - b

print(f"a = {a:.20f}")
print(f"b = {b:.20f}")
print(f"a - b = {res}")

a = 1.23456788063049316406
b = 1.23456776142120361328
a - b = 1.1920928955078125e-07


### Example: The rate of change

A common task is to find the rate of change of a measurement. This is discussed later in *finite difference* but naively one would expect: 

$$
\Delta T = T_{final} - T_{initial}
$$ (temp_change)

which generates spurious results when $T_{final} \approx T_{initial}$.

### E.g.: The quadratic formula



$$
x = \frac{-b\pm \sqrt{b^2-4ac}}{2a}
$$ (quadratic_formula)

if $4ac \ll b^2$ this will result in *subtractive cancellation* in the numerator.

Solve the roots of $$ x^2 + 10^8x+1$$

Recall the quadratic formula $$ x = \frac{-b\pm \sqrt{b^2-4ac}}{2a} $$

In [42]:
a,b,c = 1, 1e8, 1
desc = np.sqrt(b**2 - 4*a*c)
print('The roots are: ', (-b+desc)/(2*a), 'and', (-b-desc)/(2*a))

The roots are:  -7.450580596923828e-09 and -100000000.0


What else could we do? Where does the problem occur? 

The problem occurs when $b>>ac$. The descriminant becomes ~b and the numerator can suffer subtractive cancellation on either the plus or negative, depending on the sign of $b$. 

A numerically stabilized method accounts for the sign of $b$; $sign(b)=0,1$ if $b$ is positive or negative, and the relationship $x_1 \cdot x_2 = c/a$

In [45]:
q = -0.5 * (b + np.sign(b) * np.sqrt(b**2 - 4*a*c))
print('The roots are: ', q/a, 'and', c/q)

The roots are:  -100000000.0 and -1e-08
